# FlowFigTabMiner - Colab Setup & Test

One-click pipeline test on Google Colab.

**Requirements**: GPU runtime (T4 recommended). The GitHub repo must be **public**.

## 1. Clone & Install

In [ ]:
!git clone https://github.com/wzjeh/FlowFigTabMiner.git
%cd FlowFigTabMiner
!ls README.md requirements.txt config.yaml .env.example

In [ ]:
# Install PaddlePaddle GPU
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# Install all other dependencies
!pip install -q ultralytics==8.4.2 paddleocr paddlex
!pip install -q transformers==4.57.3 tokenizers==0.22.2 timm==0.4.12 einops==0.8.1
!pip install -q OpenNMT-py==2.2.0 SmilesPE==0.0.3 albumentations==1.1.0
!pip install -q rdkit scipy scikit-learn pypdfium2 dashscope PyYAML PyMuPDF
print('\n=== Installation Complete ===')

## 2. Download Models

In [ ]:
# Download 5 custom YOLO models from HuggingFace
!pip install -q huggingface_hub
!huggingface-cli download wyzhaoc/YOLO11 --local-dir models/hf_yolo11

import os, shutil
for src, dst in {
    'models/hf_yolo11/fig-seg/best.pt': 'models/yolo11m-fig-seg-0207-nobreaknocharttext/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/fig-sca/best.pt': 'models/yolo11m-fig-scatter-0208/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-seg/best.pt': 'models/yolo11m-tab-seg-0209-white/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-mol/best.pt': 'models/yolo11s-tab-molecule-0207/runs/detect/train/weights/best.pt',
    'models/hf_yolo11/tab-scheme-seg/best.pt': 'models/tab-scheme-seg/best.pt',
}.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f'  OK: {dst}')
print('\nYOLO models ready.')

In [ ]:
# Download MolNexTR (1.06 GB) from official HuggingFace dataset
# Source: Chen et al., J. Cheminf. 2024 (https://doi.org/10.1186/s13321-024-00926-w)
import os
if not os.path.exists('models/molnextr_model_best.pth'):
    !huggingface-cli download CYF200127/MolNexTR molnextr_best.pth --local-dir models/ --repo-type dataset
    if os.path.exists('models/molnextr_best.pth'):
        os.rename('models/molnextr_best.pth', 'models/molnextr_model_best.pth')
        print(f'MolNexTR downloaded: {os.path.getsize("models/molnextr_model_best.pth") / 1e9:.2f} GB')
    else:
        print('Download failed. Manual upload: https://huggingface.co/datasets/CYF200127/MolNexTR')
else:
    print('MolNexTR already exists.')

## 3. Configure API Key & LLM Model

In [ ]:
import os

# Paste your DashScope API key (optional — Steps 1-4 work without it)
QWEN_API_KEY = ''  # <-- paste your key here

# Create .env
os.environ['QWEN_API_KEY'] = QWEN_API_KEY
with open('.env', 'w') as f:
    f.write(f'QWEN_API_KEY={QWEN_API_KEY}\n')

# Switch LLM model (optional)
!sed -i 's/qwen3.5-plus/qwen3.6-plus/g' config.yaml
!grep model_name config.yaml

print('\nConfig ready.' + (' API key set.' if QWEN_API_KEY else ' No API key — Step 5 will be skipped.'))

## 4. Upload PDF & Run Pipeline

In [ ]:
from google.colab import files
print('Upload a flow chemistry PDF:')
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
!mkdir -p data/input
import shutil
shutil.move(pdf_name, f'data/input/{pdf_name}')
print(f'Uploaded: data/input/{pdf_name}')

In [ ]:
!python -m src.pipeline.main "data/input/{pdf_name}"

## 5. Check Results

In [ ]:
import glob, json, os
basename = pdf_name.rsplit('.', 1)[0]

figs = glob.glob(f'data/intermediate/{basename}/figures/*.png')
tabs = glob.glob(f'data/intermediate/{basename}/tables/*_extracted.csv')
evidences = glob.glob(f'data/intermediate/{basename}/**/*evidence*.json', recursive=True)

print(f'Figures detected: {len(figs)}')
print(f'Tables extracted: {len(tabs)}')
print(f'Evidence files:   {len(evidences)}')

for t in tabs:
    print(f'\n--- {os.path.basename(t)} ---')
    with open(t) as f:
        for i, line in enumerate(f):
            if i < 5: print(line.rstrip())

final = f'data/final_output/{basename}_normalized.json'
if os.path.exists(final):
    records = json.load(open(final))
    print(f'\nFinal records: {len(records)}')
else:
    print('\nNo final output (Step 5 LLM may have been skipped)')

print('\n=== Pipeline Verification Complete ===')